# Notebook 57 — Telemetry

**Module:** `multigen.telemetry`

Demonstrates the full observability stack: Langfuse-style traces/generations/sessions/scores,
distributed W3C tracing, Prometheus-format metrics, structured JSON logging, epistemic
propagation, multi-tenant reporting, and the in-memory span exporter.

All examples use mock data and require no live LLM or external service.

**Topics covered:**
1. Setup
2. Creating a Trace (Langfuse-style)
3. Recording a Generation with cost
4. Sessions and multi-turn
5. Scoring an observation
6. Distributed Tracing (W3C TraceContext)
7. AgentSpan with AI-specific attributes
8. Metrics (Prometheus-style)
9. Structured Logging with correlation
10. Epistemic propagation
11. Tenant observability report
12. InMemorySpanExporter

## 1. Setup

In [ ]:
import importlib.util, pathlib, sys

def load_module(name):
    path = pathlib.Path('../sdk/multigen') / f'{name}.py'
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

tel = load_module('telemetry')

from multigen.telemetry import ObservabilityManager, get_telemetry

print('ObservabilityManager:', ObservabilityManager)
print('get_telemetry:', get_telemetry)

## 2. Creating a Trace (Langfuse-style)

`ObservabilityManager.create()` instantiates the facade.  
The `obs.trace()` context manager creates a `Trace` object — the top-level container
for a single workflow execution, equivalent to a Langfuse Trace.

In [ ]:
import json
from multigen.telemetry import ObservabilityManager

obs = ObservabilityManager.create("my-service")

with obs.trace("research-workflow", tenant_id="acme") as trace:
    # Simulate work inside the workflow
    trace.metadata["query"] = "What is the market size of generative AI?"
    trace.tags.append("research")
    trace.output = {
        "summary": "Generative AI market is projected to reach $1.3T by 2032.",
        "sources": 5,
    }

print("trace.status :", trace.status)
print("trace.tenant_id:", trace.tenant_id)
print()
print(json.dumps(trace.to_dict(), indent=2, default=str))

## 3. Recording a Generation with cost

`obs.generation()` creates a `Generation` record (an individual LLM call).  
After the (mock) call, `.finish()` records the output, token counts, and automatically
computes `cost_usd` from the built-in price table.

In [ ]:
import json
from multigen.telemetry import ObservabilityManager

obs = ObservabilityManager.create("my-service")

with obs.trace("research-workflow", tenant_id="acme") as trace:
    gen = obs.generation(
        trace.trace_id,
        "gpt-4o-call",
        "gpt-4o",
        input=[
            {"role": "system", "content": "You are a research analyst."},
            {"role": "user",   "content": "Summarise the generative AI market."},
        ],
    )

    # Simulate the LLM call completing
    gen.finish(
        output="Generative AI is a rapidly growing market projected at $1.3T by 2032.",
        input_tokens=150,
        output_tokens=80,
    )

print(f"generation_id : {gen.generation_id}")
print(f"model         : {gen.model}")
print(f"input_tokens  : {gen.input_tokens}")
print(f"output_tokens : {gen.output_tokens}")
print(f"cost_usd      : {gen.cost_usd:.6f}  (${gen.cost_usd*100:.4f} cents)")
print(f"latency_ms    : {gen.latency_ms:.2f} ms")
print()
print(json.dumps(gen.to_dict(), indent=2, default=str))

## 4. Sessions and multi-turn

`obs.session()` groups multiple traces under a single user/conversation session —
equivalent to a Langfuse Session. Traces created inside automatically register
their `trace_id` on `session.trace_ids`.

In [ ]:
import json
from multigen.telemetry import ObservabilityManager

obs = ObservabilityManager.create("my-service")

with obs.session(user_id="alice", tenant_id="acme") as session:
    # Turn 1
    with obs.trace("turn-1", session_id=session.session_id, tenant_id="acme") as t1:
        t1.input  = "What is RAG?"
        t1.output = "RAG stands for Retrieval-Augmented Generation."

    # Turn 2
    with obs.trace("turn-2", session_id=session.session_id, tenant_id="acme") as t2:
        t2.input  = "Give me an example."
        t2.output = "A chatbot that retrieves docs before answering."

print("session_id  :", session.session_id)
print("user_id     :", session.user_id)
print("tenant_id   :", session.tenant_id)
print("trace_ids   :", session.trace_ids)
print("trace_count :", len(session.trace_ids))
print()
print(json.dumps(session.to_dict(), indent=2, default=str))

## 5. Scoring an observation

`obs.score()` attaches a named numeric score to any observation ID
(trace, generation, or span). Useful for evaluation metrics like relevance,
faithfulness, or toxicity.

In [ ]:
import json
from multigen.telemetry import ObservabilityManager

obs = ObservabilityManager.create("my-service")

with obs.trace("scored-workflow", tenant_id="acme") as trace:
    gen = obs.generation(
        trace.trace_id, "answer-gen", "gpt-4o",
        input=[{"role": "user", "content": "Explain transformers."}],
    )
    gen.finish(output="Transformers use self-attention to process sequences in parallel.",
               input_tokens=20, output_tokens=15)

# Score the generation
score = obs.score(
    trace.trace_id,
    "relevance",
    0.92,
    observation_type="trace",
    source="model",
    comment="Answer is highly relevant to the question.",
    tenant_id="acme",
)

print("score_id         :", score.score_id)
print("name             :", score.name)
print("value            :", score.value)
print("source           :", score.source)
print("observation_type :", score.observation_type)
print()
print(json.dumps(score.to_dict(), indent=2, default=str))

# Retrieve all scores for this observation
all_scores = obs.scores.for_observation(trace.trace_id)
print(f"\nTotal scores on trace: {len(all_scores)}")

## 6. Distributed Tracing (W3C TraceContext)

`TraceContext` implements the W3C Trace Context specification.  
`inject(headers)` encodes context into HTTP headers; `extract(headers)` recovers
it on the receiving end, enabling cross-agent / cross-service trace propagation.

In [ ]:
from multigen.telemetry import TraceContext

# Create a root context (e.g. at the orchestrator)
root = TraceContext.new_root()
print("Root trace_id   :", root.trace_id)
print("Root span_id    :", root.span_id)
print("Root traceparent:", root.to_traceparent())

# Create a child span (e.g. passed to a downstream agent)
child = root.child()
print("\nChild span_id       :", child.span_id)
print("Child parent_span_id:", child.parent_span_id)
print("Child trace_id      :", child.trace_id, "  (same as root)")

# Inject into outgoing HTTP headers
headers: dict = {"Content-Type": "application/json"}
child.inject(headers)
print("\nHeaders after inject:")
for k, v in headers.items():
    print(f"  {k}: {v}")

# Extract on the receiving side
recovered = TraceContext.extract(headers)
print("\nRecovered from headers:")
print("  trace_id:", recovered.trace_id)
print("  span_id :", recovered.span_id)
print("  flags   :", recovered.flags)
print("  Same trace_id as root?", recovered.trace_id == root.trace_id)

## 7. AgentSpan with AI-specific attributes

`AgentSpan` extends `Span` with AI-native attributes: token usage, confidence,
goal drift, and security events. Create one via `tracer.start_span()` with
`agent_id` / `agent_name` / `model` arguments.

In [ ]:
import json
from multigen.telemetry import ObservabilityManager, SpanStatus

obs = ObservabilityManager.create("my-service")

span = obs.tracer.start_span(
    "research-agent",
    agent_id="agent-42",
    agent_name="ResearchAgent",
    model="gpt-4o",
    tenant_id="acme",
)

# Record AI-specific attributes
span.record_token_usage(100, 50)
span.record_confidence(0.85)
span.record_goal_drift(0.12, reason="topic shifted from AI ethics to AI capabilities")
span.add_security_event("prompt_injection", severity="high",
                        details={"payload": "Ignore previous instructions"})

span.set_status(SpanStatus.OK)
span.end()

d = span.to_dict()
print("name            :", d["name"])
print("agent_id        :", span.agent_id)
print("model           :", span.model)
print("input_tokens    :", span.input_tokens)
print("output_tokens   :", span.output_tokens)
print("confidence      :", span.confidence)
print("goal_drift_score:", span.goal_drift_score)
print("\nevents:")
for ev in d["events"]:
    print("  ", json.dumps(ev, default=str))
print("\nattributes (subset):")
for k, v in d["attributes"].items():
    print(f"  {k}: {v}")

## 8. Metrics (Prometheus-style)

`obs.meter` is a `Meter` factory that creates `Counter`, `Histogram`, and `Summary`
instruments and registers them in the global `MetricRegistry`.  
`MetricRegistry.to_prometheus_text()` renders the full Prometheus exposition format.

In [ ]:
from multigen.telemetry import ObservabilityManager, MetricRegistry

# Reset registry to avoid duplicate instruments from previous cells
MetricRegistry.reset()

obs = ObservabilityManager.create("my-service")

# Counter — counts LLM API requests
llm_requests = obs.meter.counter(
    "llm_requests",
    description="Number of LLM API requests",
)
llm_requests.add(1, labels={"model": "gpt-4o", "tenant": "acme"})
llm_requests.add(3, labels={"model": "gpt-4o", "tenant": "acme"})
llm_requests.add(2, labels={"model": "gpt-4o-mini", "tenant": "beta"})

# Histogram — tracks request latency
latency_hist = obs.meter.histogram(
    "request_latency_ms",
    description="End-to-end request latency",
    buckets=(10, 50, 100, 250, 500, 1000, float("inf")),
)
for ms in [45, 120, 88, 310, 55, 720, 95]:
    latency_hist.observe(ms, labels={"tenant": "acme"})

# Summary — tracks token count distribution
token_summary = obs.meter.summary(
    "token_count",
    description="Tokens per generation",
)
for t in [50, 80, 120, 200, 65, 95, 150, 40, 300, 75]:
    token_summary.observe(t, labels={"model": "gpt-4o"})

print("Counter value (gpt-4o / acme):", llm_requests.get(labels={"model": "gpt-4o", "tenant": "acme"}))
print("P50 tokens                   :", token_summary.quantile(0.50, labels={"model": "gpt-4o"}))
print("P95 tokens                   :", token_summary.quantile(0.95, labels={"model": "gpt-4o"}))
print()
print("--- Prometheus text output (first 50 lines) ---")
prom_text = MetricRegistry.to_prometheus_text()
for line in prom_text.splitlines()[:50]:
    print(line)

## 9. Structured Logging with correlation

`StructuredLogger` emits JSON log records automatically correlated with the
active trace/span/tenant context via `CorrelationContext`.  
Each record exposes `.to_json()` for structured log shipping.

In [ ]:
import json
from multigen.telemetry import ObservabilityManager, set_correlation, StructuredLogger, LogLevel

obs = ObservabilityManager.create("my-service")

# Set correlation context manually (normally set by obs.trace() context manager)
token = set_correlation(
    trace_id="abc123def456",
    span_id="span99",
    tenant_id="acme",
    agent_id="agent-1",
    workflow_id="wf-77",
)

# Log via the shared ObservabilityManager logger
obs.tracer._logger = obs.logger  # expose logger (already on obs.logger)
obs.logger.info("Fetching documents from vector store",
                retrieval_k=5, query_length=42)
obs.logger.warn("Low confidence score detected",
                confidence=0.62, threshold=0.70)
obs.logger.error("Downstream model timeout",
                 model="gpt-4o", timeout_ms=5000)

# Direct StructuredLogger usage
logger = StructuredLogger("research-agent", level=LogLevel.DEBUG)
logger.info("Generation complete", tokens_out=80, cost_usd=0.0012)

# Inspect buffered records
records = obs.logger.buffer.tail(10)
print(f"Buffered log records: {len(records)}")
print()
for rec in records:
    parsed = json.loads(rec.to_json())
    print(json.dumps(parsed, indent=2, default=str))
    print("---")

# Clean up correlation context
from multigen.telemetry import _correlation_ctx
_correlation_ctx.reset(token)

## 10. Epistemic propagation

`EpistemicAttribute` is a novel Multigen primitive that attaches confidence and
propagated uncertainty to any Span. The static `propagate()` method applies the
rule: `effective_confidence = child_confidence × parent_effective_confidence`.

In [ ]:
from multigen.telemetry import EpistemicAttribute, ObservabilityManager

obs = ObservabilityManager.create("my-service")

# Parent node: high-confidence retrieval
parent = EpistemicAttribute(
    confidence=0.9,
    uncertainty_sources=["limited corpus coverage"],
    propagated_uncertainty=0.0,
    effective_confidence=0.9,
    evidence_quality="high",
)

# Child node: lower-confidence inference step
child = EpistemicAttribute.propagate(parent, 0.8,
                                     child_sources=["ambiguous phrasing"])

print("Parent:")
print(f"  confidence          : {parent.confidence}")
print(f"  effective_confidence: {parent.effective_confidence}")
print(f"  propagated_uncert.  : {parent.propagated_uncertainty}")

print("\nChild (propagated from parent):")
print(f"  own confidence      : {child.confidence}")
print(f"  effective_confidence: {child.effective_confidence}  "
      f"(= {child.confidence} × {parent.effective_confidence})")
print(f"  propagated_uncert.  : {child.propagated_uncertainty}  "
      f"(= 1 - {parent.effective_confidence})")
print(f"  uncertainty_sources : {child.uncertainty_sources}")

# Attach to a span
with obs.tracer.span("inference-step", tenant_id="acme") as span:
    child.attach_to_span(span)

print("\nSpan epistemic attributes:")
for k, v in span.attributes.items():
    if k.startswith("epistemic"):
        print(f"  {k}: {v}")

## 11. Tenant observability report

`obs.tenant_report(tenant_id)` aggregates all spans, generations, sessions, and
scores for a given tenant into a `TenantObservabilityReport` dataclass with
latency percentiles, error rates, cost, confidence, goal drift, and more.

In [ ]:
import json, time
from multigen.telemetry import ObservabilityManager

obs = ObservabilityManager.create("reporting-service")
TENANT = "acme"

# --- Trace 1: research workflow ---
with obs.trace("research-wf", tenant_id=TENANT, tags=["research"]) as t1:
    g1 = obs.generation(t1.trace_id, "fetch-context", "gpt-4o",
                        input=[{"role": "user", "content": "Summarise AI trends"}],
                        tenant_id=TENANT)
    g1.finish(output="AI is trending toward multimodal systems.",
              input_tokens=200, output_tokens=120)
    obs.score(t1.trace_id, "relevance", 0.95,
              observation_type="trace", source="model", tenant_id=TENANT)

    with obs.tracer.span("vector-search", tenant_id=TENANT) as s1:
        s1.set_attribute("db.hits", 8)

# --- Trace 2: synthesis workflow ---
with obs.trace("synthesis-wf", tenant_id=TENANT, tags=["synthesis"]) as t2:
    g2 = obs.generation(t2.trace_id, "synthesise", "gpt-4o",
                        input=[{"role": "user", "content": "Draft executive summary"}],
                        tenant_id=TENANT)
    g2.finish(output="Executive summary: AI is reshaping every industry.",
              input_tokens=350, output_tokens=200)
    obs.score(t2.trace_id, "faithfulness", 0.88,
              observation_type="trace", source="model", tenant_id=TENANT)

    with obs.tracer.span("rerank", tenant_id=TENANT) as s2:
        s2.set_attribute("rerank.top_k", 3)

# --- Trace 3: validation workflow ---
with obs.trace("validation-wf", tenant_id=TENANT) as t3:
    g3 = obs.generation(t3.trace_id, "validate", "gpt-4o-mini",
                        input=[{"role": "user", "content": "Check for hallucinations"}],
                        tenant_id=TENANT)
    g3.finish(output="No hallucinations detected.",
              input_tokens=80, output_tokens=30)
    obs.score(t3.trace_id, "relevance", 0.91,
              observation_type="trace", source="rule", tenant_id=TENANT)

# --- Produce report ---
report = obs.tenant_report(TENANT)
print(json.dumps(report.to_dict(), indent=2, default=str))

## 12. InMemorySpanExporter

`InMemorySpanExporter` is the default exporter attached to every
`ObservabilityManager`. It stores finished spans in memory for inspection,
filtering, and testing without any external infrastructure.

In [ ]:
import json
from multigen.telemetry import ObservabilityManager, InMemorySpanExporter

# Attach a dedicated exporter so we can inspect independently
exporter = InMemorySpanExporter()
obs = ObservabilityManager.create("my-service")
obs.add_exporter(exporter)

# Run a few spans across two tenants
with obs.tracer.span("retrieve",  tenant_id="acme") as s1:
    s1.set_attribute("db.type", "vector")

with obs.tracer.span("rank",      tenant_id="acme") as s2:
    s2.set_attribute("rank.algo", "bm25")

with obs.tracer.span("summarise", tenant_id="beta") as s3:
    s3.set_attribute("model", "claude-sonnet-4-6")

with obs.tracer.span("validate",  tenant_id="acme") as s4:
    s4.set_attribute("checks", 3)

# Retrieve all finished spans
all_spans = exporter.get_finished_spans()
print(f"Total finished spans: {len(all_spans)}")
print()

# Filter by tenant
acme_spans = exporter.filter_by_tenant("acme")
beta_spans  = exporter.filter_by_tenant("beta")
print(f"Spans for tenant 'acme': {len(acme_spans)}")
print(f"Spans for tenant 'beta': {len(beta_spans)}")
print()

print("acme span names:", [s.name for s in acme_spans])
print("beta span names:", [s.name for s in beta_spans])
print()

# Show full dict for each acme span
print("--- acme spans detail ---")
for span in acme_spans:
    d = span.to_dict()
    print(f"  [{d['name']:12s}] status={d['status']:5s}  "
          f"duration={d['duration_ms']:.2f}ms  "
          f"attrs={d['attributes']}")